In [4]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from pyteomics import mzml, fasta, auxiliary
import itertools
import copy
import time
import os
from os import path, listdir

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Fasta

In [13]:
output_file = './fasta/human_ecoli_yeast_13072026.fasta'

with open('./fasta/UP000005640_9606_human_10_07_2026.fasta', mode='r') as hf, \
     open('./fasta/UP000000625_83333_ecoli_10_07_2026.fasta', mode='r') as ef, \
     open('./fasta/UP000002311_559292_yeast_10_07_2026.fasta', mode='r') as yf :
    input_files = [hf, ef, yf]
    with open(output_file, mode='w') as outf :
        for i in input_files :
            records = fasta.read(i)
            fasta.write(records, outf)

In [14]:
with open('./fasta/UP000005640_9606_human_10_07_2026.fasta', mode='r') as hf, \
     open('./fasta/UP000000625_83333_ecoli_10_07_2026.fasta', mode='r') as ef, \
     open('./fasta/UP000002311_559292_yeast_10_07_2026.fasta', mode='r') as yf :
    input_files = [hf, ef, yf]
    
    for i in input_files :
        j = 0
        for _ in fasta.read(i) :
            j += 1
        print(j)

20652
4403
6066


In [16]:
with open(output_file, mode='r') as i :
    j = 0
    for _ in fasta.read(i) :
        j += 1
print(j)

31121


In [ ]:
customdecoypath = './fasta/human_ecoli_yeast_13072026_shuffled.fasta'
output_file = './fasta/human_ecoli_yeast_13072026.fasta'

with open(output_file, mode='r') as i :
    with open(customdecoypath, mode='w') as o :
        fasta.write_decoy_db(source=i, output=o, mode='shuffle', prefix='DECOY_')
with open(customdecoypath, mode='r') as o :
    j = 0
    for _ in fasta.read(o) :
        j += 1
print(j)

In [79]:
customdecoypath = './fasta/sprot_ecoli_ups_15072026_shuffled.fasta'
input_file = './fasta/sprot_ecoli_ups.fasta'
with open(input_file, mode='r') as i :
    with open(customdecoypath, mode='w') as o :
        fasta.write_decoy_db(source=i, output=o, mode='shuffle', prefix='DECOY_')
with open(customdecoypath, mode='r') as o :
    j = 0
    for _ in fasta.read(o) :
        j += 1
print(j)

8958


# DIA-NN 2.3.1 LFQbench Orbitrap

In [5]:
diann_path = '~/tools/diann-2.3.1/diann-linux'

In [6]:
command = []
command.append(diann_path)
# fold = mzml_path
args = ['--threads',  '20', '--verbose', '5', '--out', './search_results/diann231_LFQbench/report.tsv', 
     '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
     '--out-lib', './search_results/diann231_LFQbench/lib.tsv',
     '--fasta', './fasta/human_ecoli_yeast_13072026.fasta', '--met-excision', 
     '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', '--predictor',
     '--fasta-search', '--gen-spec-lib',]
command = command + args
print(' '.join(command))

~/tools/diann-2.3.1/diann-linux --threads 20 --verbose 5 --out ./search_results/diann231_LFQbench/report.tsv --qvalue 0.01 --matrices --temp ./search_results/tmp --out-lib ./search_results/diann231_LFQbench/lib.tsv --fasta ./fasta/human_ecoli_yeast_13072026.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --predictor --fasta-search --gen-spec-lib


In [22]:
!~/tools/diann-2.3.1/diann-linux --threads 20 --verbose 5 --out ./search_results/diann231_LFQbench/report.tsv --qvalue 0.01 --matrices --temp ./search_results/tmp --out-lib ./search_results/diann231_LFQbench/lib.tsv --fasta ./fasta/human_ecoli_yeast_13072026.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --predictor --fasta-search --gen-spec-lib


DIA-NN 2.3.1 Academia  (Data-Independent Acquisition by Neural Networks)
Compiled on Dec  5 2025 05:02:42
Current date and time: Mon Jul 13 15:49:59 2026
Logical CPU cores: 256
Thread number set to 20
Output will be filtered at 0.01 FDR
Precursor/protein x samples expression level matrices will be saved along with the main report
N-terminal methionine excision enabled
In silico digest will involve cuts at K*,R*
Heuristic protein grouping will be used, to reduce the number of protein groups obtained; this mode is recommended for benchmarking protein ID numbers, GO/pathway and system-scale analyses
When generating an empirical library, empirical spectra will be added to it only if they are high quality
Deep learning will be used to generate a new in silico spectral library from peptides provided
DIA-NN will carry out FASTA digest for in silico lib generation
A spectral library will be generated

0 files will be processed
[0:00] Loading FASTA /home/lerost/DIA_tools_manuscript/fasta/human

In [7]:
command = []
command.append(diann_path)
fold = './input_files/LFQbench_orbitrap'
for f in sorted(os.listdir(fold)) :
    if f.endswith('.mzML') and 'Orbitrap' in f and 'Condition' in f :
        command.append('--f')
        command.append(os.path.join(fold, f))
# command.append('--f')
# command.append(file)
args = ['--threads',  '20', '--verbose', '5', '--out', './search_results/diann231_LFQbench/report.tsv', 
         '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
         '--out-lib', './search_results/diann231_LFQbench/lib.tsv',
         '--fasta', './fasta/human_ecoli_yeast_13072026.fasta', '--met-excision', 
         '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', 
         '--lib', './search_results/diann231_LFQbench/lib.predicted.speclib',
         '--report-decoys'
       ]
command = command + args
print(' '.join(command))

~/tools/diann-2.3.1/diann-linux --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02.mzML --f ./input_files/LFQbenc

In [ ]:
!~/tools/diann-2.3.1/diann-linux --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Alpha_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Beta_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_A_Sample_Gamma_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Alpha_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Beta_03.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_01.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_02.mzML --f ./input_files/LFQbench_orbitrap/LFQ_Orbitrap_AIF_Condition_B_Sample_Gamma_03.mzML --threads 20 --verbose 5 --out ./search_results/diann231_LFQbench/report.tsv --qvalue 0.01 --matrices --temp ./search_results/tmp --out-lib ./search_results/diann231_LFQbench/lib.tsv --fasta ./fasta/human_ecoli_yeast_13072026.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --lib ./search_results/diann231_LFQbench/lib.predicted.speclib --report-decoys


DIA-NN 2.3.1 Academia  (Data-Independent Acquisition by Neural Networks)
Compiled on Dec  5 2025 05:02:42
Current date and time: Mon Jul 13 16:18:28 2026
Logical CPU cores: 256
Thread number set to 20
Output will be filtered at 0.01 FDR
Precursor/protein x samples expression level matrices will be saved along with the main report
N-terminal methionine excision enabled
In silico digest will involve cuts at K*,R*
Heuristic protein grouping will be used, to reduce the number of protein groups obtained; this mode is recommended for benchmarking protein ID numbers, GO/pathway and system-scale analyses
When generating an empirical library, empirical spectra will be added to it only if they are high quality
DIA-NN will report decoy PSMs in its main report
DIA-NN will automatically optimise the mass accuracy for the first run of the experiment, use this mode for preliminary analyses only

18 files will be processed
[0:00] Loading spectral library /home/lerost/DIA_tools_manuscript/search_resul

# DIA-NN 2.3.1 LFQbench Bruker timsTOF Pro

In [8]:
diann_path = '~/tools/diann-2.3.1/diann-linux'

In [9]:
command = []
command.append(diann_path)
# fold = mzml_path
args = ['--threads',  '20', '--verbose', '5', '--out', './search_results/diann231_LFQbenchTOF/report.tsv', 
     '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
     '--out-lib', './search_results/diann231_LFQbenchTOF/lib.tsv',
     '--fasta', './fasta/human_ecoli_yeast_13072026.fasta', '--met-excision', 
     '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', '--predictor',
     '--fasta-search', '--gen-spec-lib',]
command = command + args
print(' '.join(command))

~/tools/diann-2.3.1/diann-linux --threads 20 --verbose 5 --out ./search_results/diann231_LFQbenchTOF/report.tsv --qvalue 0.01 --matrices --temp ./search_results/tmp --out-lib ./search_results/diann231_LFQbenchTOF/lib.tsv --fasta ./fasta/human_ecoli_yeast_13072026.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --predictor --fasta-search --gen-spec-lib


In [103]:
!~/tools/diann-2.3.1/diann-linux --threads 20 --verbose 5 --out ./search_results/diann231_LFQbenchTOF/report.tsv --qvalue 0.01 --matrices --temp ./search_results/tmp --out-lib ./search_results/diann231_LFQbenchTOF/lib.tsv --fasta ./fasta/human_ecoli_yeast_13072026.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --predictor --fasta-search --gen-spec-lib


DIA-NN 2.3.1 Academia  (Data-Independent Acquisition by Neural Networks)
Compiled on Dec  5 2025 05:02:42
Current date and time: Mon Jul 27 12:27:53 2026
Logical CPU cores: 256
Thread number set to 20
Output will be filtered at 0.01 FDR
Precursor/protein x samples expression level matrices will be saved along with the main report
N-terminal methionine excision enabled
In silico digest will involve cuts at K*,R*
Heuristic protein grouping will be used, to reduce the number of protein groups obtained; this mode is recommended for benchmarking protein ID numbers, GO/pathway and system-scale analyses
When generating an empirical library, empirical spectra will be added to it only if they are high quality
Deep learning will be used to generate a new in silico spectral library from peptides provided
DIA-NN will carry out FASTA digest for in silico lib generation
A spectral library will be generated

0 files will be processed
[0:00] Loading FASTA /home/lerost/DIA_tools_manuscript/fasta/human

In [10]:
command = []
command.append(diann_path)
fold = './input_files/LFQbench_timsTOF'
for f in sorted(os.listdir(fold)) :
    if f.endswith('.d') and 'diaPASEF' in f and 'Condition' in f :
        command.append('--f')
        command.append(os.path.join(fold, f))
# command.append('--f')
# command.append(file)
args = ['--threads',  '20', '--verbose', '5', '--out', './search_results/diann231_LFQbenchTOF/report.tsv', 
         '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
         '--out-lib', './search_results/diann231_LFQbenchTOF/lib.tsv',
         '--fasta', './fasta/human_ecoli_yeast_13072026.fasta', '--met-excision', 
         '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', 
         '--lib', './search_results/diann231_LFQbenchTOF/lib.predicted.speclib',
         '--report-decoys'
       ]
command = command + args
print(' '.join(command))

~/tools/diann-2.3.1/diann-linux --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alp

In [ ]:
!~/tools/diann-2.3.1/diann-linux --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Alpha_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Beta_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_A_Sample_Gamma_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Alpha_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Beta_03.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_01.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_02.d --f ./input_files/LFQbench_timsTOF/LFQ_timsTOFPro_diaPASEF_Condition_B_Sample_Gamma_03.d --threads 20 --verbose 5 --out ./search_results/diann231_LFQbenchTOF/report.tsv --qvalue 0.01 --matrices --temp ./search_results/tmp --out-lib ./search_results/diann231_LFQbenchTOF/lib.tsv --fasta ./fasta/human_ecoli_yeast_13072026.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --lib ./search_results/diann231_LFQbenchTOF/lib.predicted.speclib --report-decoys


DIA-NN 2.3.1 Academia  (Data-Independent Acquisition by Neural Networks)
Compiled on Dec  5 2025 05:02:42
Current date and time: Wed Jul 29 15:24:18 2026
Logical CPU cores: 256
Thread number set to 20
Output will be filtered at 0.01 FDR
Precursor/protein x samples expression level matrices will be saved along with the main report
N-terminal methionine excision enabled
In silico digest will involve cuts at K*,R*
Heuristic protein grouping will be used, to reduce the number of protein groups obtained; this mode is recommended for benchmarking protein ID numbers, GO/pathway and system-scale analyses
When generating an empirical library, empirical spectra will be added to it only if they are high quality
DIA-NN will report decoy PSMs in its main report
DIA-NN will automatically optimise the mass accuracy for the first run of the experiment, use this mode for preliminary analyses only

18 files will be processed
[0:00] Loading spectral library /home/lerost/DIA_tools_manuscript/search_resul

# UPS-ecoli MSFragger

In [88]:
input_dir = './input_files/PXD026600/'
# RD139_Wide_UPS1_0_1fmol_inj1.mzML
concs = set()
for file in listdir(input_dir) :
    conc = file.split('UPS1_')[-1].split('_inj')[0]
    concs.add(conc)

concs = sorted(list(concs), key=lambda x: float(x.split('fmol')[0].replace('_', '.')) , reverse=True)
print(concs)
pairwise_concs = [conc1 + '_vs_' + conc2 for conc1 in concs for conc2 in concs if conc1 != conc2 and float(conc1.split('fmol')[0].replace('_', '.')) > float(conc2.split('fmol')[0].replace('_', '.'))]
print(pairwise_concs)

['50fmol', '25fmol', '10fmol', '5fmol', '2_5fmol', '1fmol', '0_25fmol', '0_1fmol']
['50fmol_vs_25fmol', '50fmol_vs_10fmol', '50fmol_vs_5fmol', '50fmol_vs_2_5fmol', '50fmol_vs_1fmol', '50fmol_vs_0_25fmol', '50fmol_vs_0_1fmol', '25fmol_vs_10fmol', '25fmol_vs_5fmol', '25fmol_vs_2_5fmol', '25fmol_vs_1fmol', '25fmol_vs_0_25fmol', '25fmol_vs_0_1fmol', '10fmol_vs_5fmol', '10fmol_vs_2_5fmol', '10fmol_vs_1fmol', '10fmol_vs_0_25fmol', '10fmol_vs_0_1fmol', '5fmol_vs_2_5fmol', '5fmol_vs_1fmol', '5fmol_vs_0_25fmol', '5fmol_vs_0_1fmol', '2_5fmol_vs_1fmol', '2_5fmol_vs_0_25fmol', '2_5fmol_vs_0_1fmol', '1fmol_vs_0_25fmol', '1fmol_vs_0_1fmol', '0_25fmol_vs_0_1fmol']


In [61]:
searchengine_dir = 'fragpipe24_ups_ecoli_msfragger'

for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    for file in listdir(input_dir) :
        if 'UPS1_'+conc1 in file or 'UPS1_'+conc2 in file :
            files.append(path.join(input_dir, file))
    files = sorted(files, key=lambda x: float(x.split('UPS1_')[-1].split('fmol')[0].replace('_', '.')) - 1/1000*int(x.split('inj')[-1][0]) , reverse=True)
    condition = [file.split('UPS1_')[-1].split('_inj')[0] for file in files]
    reps = [file.split('_inj')[-1][0] for file in files]
    fmt = ['DIA' for _ in files]
    m_df = pd.DataFrame([files, condition, reps, fmt]).T
    m_df.to_csv('./search_results/'+searchengine_dir+'/'+pair+'/fragpipe-files.fp-manifest', sep='\t', index=False, header=False)


In [72]:
for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    workdir = path.join('./search_results/', searchengine_dir, pair)
    fasta_path = './fasta/2026-07-14-decoys-sprot_ecoli_ups.fasta.fas'
    with open('./search_results/fragpipe24_ups_ecoli_msfragger/fragpipe.workflow', mode='r') as ref, \
         open(path.join(workdir, 'fragpipe.workflow'), mode='w') as out :
        for line in ref.readlines() :
            if line.startswith('workdir=') :
                out.write('workdir='+workdir+'\n')
            elif line.startswith('database.db-path=') :
                out.write('database.db-path='+fasta_path+'\n')
            else :
                out.write(line)

In [77]:
searchengine_dir = 'fragpipe24_ups_ecoli_msfragger_triqler'

for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    for file in listdir(input_dir) :
        if 'UPS1_'+conc1 in file or 'UPS1_'+conc2 in file :
            files.append(path.join(input_dir, file))
    files = sorted(files, key=lambda x: float(x.split('UPS1_')[-1].split('fmol')[0].replace('_', '.')) - 1/1000*int(x.split('inj')[-1][0]) , reverse=True)
    condition = [file.split('UPS1_')[-1].split('_inj')[0] for file in files]
    reps = [file.split('_inj')[-1][0] for file in files]
    fmt = ['DIA' for _ in files]
    m_df = pd.DataFrame([files, condition, reps, fmt]).T
    m_df.to_csv('./search_results/'+searchengine_dir+'/'+pair+'/fragpipe-files.fp-manifest', sep='\t', index=False, header=False)


In [80]:
for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    workdir = path.join('./search_results/', searchengine_dir, pair)
    fasta_path = './fasta/2026-07-15-decoys-sprot_ecoli_ups_15072026_shuffled.fasta.fas'
    with open('./search_results/fragpipe24_ups_ecoli_msfragger/fragpipe.workflow', mode='r') as ref, \
         open(path.join(workdir, 'fragpipe.workflow'), mode='w') as out :
        for line in ref.readlines() :
            if line.startswith('workdir=') :
                out.write('workdir='+workdir+'\n')
            elif line.startswith('database.db-path=') :
                out.write('database.db-path='+fasta_path+'\n')
            else :
                out.write(line)

# UPS-ecoli Umpire

In [73]:
searchengine_dir = 'fragpipe24_ups_ecoli_umpire'

for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    for file in listdir(input_dir) :
        if 'UPS1_'+conc1 in file or 'UPS1_'+conc2 in file :
            files.append(path.join(input_dir, file))
    files = sorted(files, key=lambda x: float(x.split('UPS1_')[-1].split('fmol')[0].replace('_', '.')) - 1/1000*int(x.split('inj')[-1][0]) , reverse=True)
    condition = [file.split('UPS1_')[-1].split('_inj')[0] for file in files]
    reps = [file.split('_inj')[-1][0] for file in files]
    fmt = ['DIA' for _ in files]
    m_df = pd.DataFrame([files, condition, reps, fmt]).T
    m_df.to_csv('./search_results/'+searchengine_dir+'/'+pair+'/fragpipe-files.fp-manifest', sep='\t', index=False, header=False)


In [76]:
for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    workdir = path.join('./search_results/', searchengine_dir, pair)
    fasta_path = './fasta/2026-07-14-decoys-sprot_ecoli_ups.fasta.fas'
    with open('./search_results/fragpipe24_ups_ecoli_umpire/fragpipe.workflow', mode='r') as ref, \
         open(path.join(workdir, 'fragpipe.workflow'), mode='w') as out :
        for line in ref.readlines() :
            if line.startswith('workdir=') :
                out.write('workdir='+workdir+'\n')
            elif line.startswith('database.db-path=') :
                out.write('database.db-path='+fasta_path+'\n')
            else :
                out.write(line)

In [81]:
searchengine_dir = 'fragpipe24_ups_ecoli_umpire_triqler'

for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    for file in listdir(input_dir) :
        if 'UPS1_'+conc1 in file or 'UPS1_'+conc2 in file :
            files.append(path.join(input_dir, file))
    files = sorted(files, key=lambda x: float(x.split('UPS1_')[-1].split('fmol')[0].replace('_', '.')) - 1/1000*int(x.split('inj')[-1][0]) , reverse=True)
    condition = [file.split('UPS1_')[-1].split('_inj')[0] for file in files]
    reps = [file.split('_inj')[-1][0] for file in files]
    fmt = ['DIA' for _ in files]
    m_df = pd.DataFrame([files, condition, reps, fmt]).T
    m_df.to_csv('./search_results/'+searchengine_dir+'/'+pair+'/fragpipe-files.fp-manifest', sep='\t', index=False, header=False)


In [82]:
for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    workdir = path.join('./search_results/', searchengine_dir, pair)
    fasta_path = './fasta/2026-07-15-decoys-sprot_ecoli_ups_15072026_shuffled.fasta.fas'
    with open('./search_results/fragpipe24_ups_ecoli_umpire/fragpipe.workflow', mode='r') as ref, \
         open(path.join(workdir, 'fragpipe.workflow'), mode='w') as out :
        for line in ref.readlines() :
            if line.startswith('workdir=') :
                out.write('workdir='+workdir+'\n')
            elif line.startswith('database.db-path=') :
                out.write('database.db-path='+fasta_path+'\n')
            else :
                out.write(line)

# UPS-ecoli DIA-NN

In [19]:
diann_path = '~/tools/diann-2.3.1/diann-linux'

In [ ]:
searchengine_dir = 'diann231_ups_ecoli'
# print(' '.join(command))

command = []
command.append(diann_path)
workdir = path.join('./search_results/', searchengine_dir)
fasta_path = './fasta/sprot_ecoli_ups.fasta'
args = ['--threads',  '20', '--verbose', '5', '--out', '{}/report.tsv'.format(workdir), 
     '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
     '--out-lib', '{}/lib.tsv'.format(workdir),
     '--fasta', fasta_path, '--met-excision', 
     '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', '--predictor',
     '--fasta-search', '--gen-spec-lib',]
command = command + args
print(' '.join(command))
command = ' '.join(command)
!$command
speclib_path = '{}/lib.predicted.speclib'.format(workdir)

for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    workdir = path.join('./search_results/', searchengine_dir, pair)
    i = 0
    command = []
    command.append(diann_path)
    conc1, conc2 = pair.split('_vs_')
    for file in listdir(input_dir) :
        if ('UPS1_'+conc1 in file or 'UPS1_'+conc2 in file) and file.endswith('.mzML') :
            command.append('--f')
            command.append(path.join(input_dir, file))
            i += 1
    print(i)
    
    args = ['--threads',  '20', '--verbose', '5', '--out', '{}/report.tsv'.format(workdir), 
             '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
             '--out-lib', '{}/lib.tsv'.format(workdir),
             '--fasta', fasta_path, '--met-excision', 
             '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', 
             '--lib', speclib_path,
             '--report-decoys'
           ]
    command = command + args
    print(' '.join(command))
    command = ' '.join(command)
    !$command
    # break

/home/lerost/tools/diann-2.3.1/diann-linux --threads 20 --verbose 5 --out ./search_results/diann231_ups_ecoli/report.tsv --qvalue 0.01 --matrices --temp /home/lerost/DIA_tools_manuscript/search_results/tmp --out-lib ./search_results/diann231_ups_ecoli/lib.tsv --fasta /home/lerost/DIA_tools_manuscript/fasta/sprot_ecoli_ups.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --predictor --fasta-search --gen-spec-lib

DIA-NN 2.3.1 Academia  (Data-Independent Acquisition by Neural Networks)
Compiled on Dec  5 2025 05:02:42
Current date and time: Thu Jul 23 17:25:18 2026
Logical CPU cores: 256
Thread number set to 20
Output will be filtered at 0.01 FDR
Precursor/protein x samples expression level matrices will be saved along with the main report
N-terminal methionine excision enabled
In silico digest will involve cuts at K*,R*
Heuristic protein grouping will be used, to reduce the number of protein groups obtained; this mode is recommended for benchmarking protein ID numbe

In [ ]:
searchengine_dir = 'diann231_ups_ecoli_triqler'
# print(' '.join(command))

command = []
command.append(diann_path)
workdir = path.join('./search_results/', searchengine_dir)
fasta_path = './fasta/sprot_ecoli_ups_15072026_shuffled.fasta'
args = ['--threads',  '20', '--verbose', '5', '--out', '{}/report.tsv'.format(workdir), 
     '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
     '--out-lib', '{}/lib.tsv'.format(workdir),
     '--fasta', fasta_path, '--met-excision', 
     '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', '--predictor',
     '--fasta-search', '--gen-spec-lib',]
command = command + args
print(' '.join(command))
command = ' '.join(command)
# !$command
speclib_path = '{}/lib.predicted.speclib'.format(workdir)

for pair in pairwise_concs :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    workdir = path.join('./search_results/', searchengine_dir, pair)
    i = 0
    command = []
    command.append(diann_path)
    conc1, conc2 = pair.split('_vs_')
    for file in listdir(input_dir) :
        if ('UPS1_'+conc1 in file or 'UPS1_'+conc2 in file) and file.endswith('.mzML') :
            command.append('--f')
            command.append(path.join(input_dir, file))
            i += 1
    print(i)
    
    args = ['--threads',  '20', '--verbose', '5', '--out', '{}/report.tsv'.format(workdir), 
             '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
             '--out-lib', '{}/lib.tsv'.format(workdir),
             '--fasta', fasta_path, '--met-excision', 
             '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', 
             '--lib', speclib_path,
             '--report-decoys'
           ]
    command = command + args
    print(' '.join(command))
    command = ' '.join(command)
    !$command
    # break

/home/lerost/tools/diann-2.3.1/diann-linux --threads 20 --verbose 5 --out ./search_results/diann231_ups_ecoli_triqler/report.tsv --qvalue 0.01 --matrices --temp /home/lerost/DIA_tools_manuscript/search_results/tmp --out-lib ./search_results/diann231_ups_ecoli_triqler/lib.tsv --fasta /home/lerost/DIA_tools_manuscript/fasta/sprot_ecoli_ups_15072026_shuffled.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --predictor --fasta-search --gen-spec-lib
6
/home/lerost/tools/diann-2.3.1/diann-linux --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_50fmol_inj2.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_25fmol_inj2.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_50fmol_inj3.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_25fmol_inj3.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_25fmol_inj1.mzML --f /home/lerost/DIA

In [110]:
searchengine_dir = 'diann231_25fmol_vs_5fmol_rerun'
# input_dir = 

command = []
command.append(diann_path)
workdir = path.join('./search_results/', searchengine_dir)
fasta_path = './fasta/sprot_ecoli_ups.fasta'
args = ['--threads',  '20', '--verbose', '5', '--out', '{}/report.tsv'.format(workdir), 
     '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
     '--out-lib', '{}/lib.tsv'.format(workdir),
     '--fasta', fasta_path, '--met-excision', 
     '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', '--predictor',
     '--fasta-search', '--gen-spec-lib',]
command = command + args
print(' '.join(command))
command = ' '.join(command)
# !$command
speclib_path = '{}/lib.predicted.speclib'.format(workdir)

for pair in ['25fmol_vs_5fmol'] :
    os.makedirs('./search_results/'+searchengine_dir+'/'+pair, exist_ok=True)
    files = []
    conc1, conc2 = pair.split('_vs_')
    workdir = path.join('./search_results/', searchengine_dir, pair)
    i = 0
    command = []
    command.append(diann_path)
    conc1, conc2 = pair.split('_vs_')
    for file in listdir(input_dir) :
        if ('UPS1_'+conc1 in file or 'UPS1_'+conc2 in file) and file.endswith('.mzML') :
            command.append('--f')
            command.append(path.join(input_dir, file))
            i += 1
    print(i)
    
    args = ['--threads',  '20', '--verbose', '5', '--out', '{}/report.tsv'.format(workdir), 
             '--qvalue', '0.01', '--matrices', '--temp', './search_results/tmp', 
             # '--out-lib', '{}/lib.tsv'.format(workdir),
             '--fasta', fasta_path, '--met-excision', 
             '--cut', 'K*,R*', '--relaxed-prot-inf', '--smart-profiling', 
             '--lib', speclib_path,
             '--report-decoys'
           ]
    command = command + args
    print(' '.join(command))
    command = ' '.join(command)
    !$command
    # break

/home/lerost/tools/diann-2.3.1/diann-linux --threads 20 --verbose 5 --out ./search_results/diann231_25fmol_vs_5fmol_rerun/report.tsv --qvalue 0.01 --matrices --temp /home/lerost/DIA_tools_manuscript/search_results/tmp --out-lib ./search_results/diann231_25fmol_vs_5fmol_rerun/lib.tsv --fasta /home/lerost/DIA_tools_manuscript/fasta/sprot_ecoli_ups.fasta --met-excision --cut K*,R* --relaxed-prot-inf --smart-profiling --predictor --fasta-search --gen-spec-lib
6
/home/lerost/tools/diann-2.3.1/diann-linux --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_5fmol_inj3.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_25fmol_inj2.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_5fmol_inj2.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_25fmol_inj3.mzML --f /home/lerost/DIA_tools_manuscript/input_files/PXD026600/RD139_Wide_UPS1_25fmol_inj1.mzML --f /home/lerost/DIA_tools_manus

# Script for flow analysis run

In [87]:
drs = [
    './search_results/fragpipe24_ups_ecoli_msfragger',
    './search_results/fragpipe24_ups_ecoli_msfragger_triqler',
    './search_results/fragpipe24_ups_ecoli_umpire',
    './search_results/fragpipe24_ups_ecoli_umpire_triqler',
]
with open('./search_results/fragpipe24_UPS-Ecoli.sh', mode='w') as script :
    for dr in drs :
        for d in listdir(dr) :
            if '_vs_' in d :
                workdir = path.join(dr, d)
                workflow = path.join(workdir, 'fragpipe.workflow')
                manifest = path.join(workdir, 'fragpipe-files.fp-manifest')
                line = '~/tools/fragpipe-24.0/bin/fragpipe --headless --workdir {} --workflow {} --manifest {} --threads 30'.format(workdir, workflow, manifest)
                script.write(line+'\n\n')